# Day 09 · 模块与高级特性

> **前置**: Day01-08 全部(重点:函数/OOP/文件 I/O)
> **关键问题**: 代码破千行后,单文件不堪重负 —— 怎么把代码**拆分到多个文件**、让重复的"计时/日志"逻辑自动复用?本节是工具箱补完课:包、生成器、上下文管理器、装饰器。

**引入:千行文件滚动条细如发丝**

抽问上节: Python 里 `class` 和 `__init__` 分别是什么?(蓝图 / 自动跑的构造函数)。再问: 一个 `utils.py` 有 2000 行,你怎么管理?引出**模块化 + 包**;再抛一个需求:"给任何函数自动计时" —— 吊足胃口。


**1. `import` / `from` / `as` 三种导入**

口诀:**`import 模块` 带前缀,`from` 直接用,`as` 改别名**。把函数/类放进 `utils.py`,同目录 `import utils` 即可。**包**(Package) = 模块的文件夹,必须含 `__init__.py`(可以为空)。


In [ ]:
import math
print(math.sqrt(16))            # 4.0,通过模块名访问

from math import sqrt
print(sqrt(16))                 # 4.0,直接用成员名

from math import factorial as fac
print(fac(5))                   # 120,别名 fac


**2. 生成器 `yield`**

普通函数 `return` 一次就结束,想**逐个产出**数据且**不占内存**,用 `yield`。生成器是惰性计算的迭代器。口诀:**`yield` 是 `return` 的"Hold"版本 —— 产出值但不退场**,下次 `next()` 从这里恢复。


In [ ]:
def 平方(limit):
    for i in range(limit):
        yield i * i             # 每次产一个值,暂停

gen = 平方(4)
print(next(gen))                # 0
print(next(gen))                # 1
print(next(gen))                # 4

# 生成器只能用一次,再遍历空了
for x in 平方(3):
    print("for:", x)            # 0 1 4


**3. 上下文管理器 `__enter__` / `__exit__`**

`with` 语句需要对象实现 `__enter__(进时自动执行)` 和 `__exit__(出时自动执行)`。`__exit__` 必须接受三个 `exc_` 参数。类比比 Day07 `with open` 自动关文件 —— 这个自动关的动作就是 `__exit__` 在做事。


In [ ]:
class Timer:
    def __enter__(self):
        import time
        self.start = time.time()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        import time
        print(f"耗时 {time.time() - self.start:.4f}s")

# with 块退出时自动调 __exit__ 计时
with Timer():
    sum(range(1000000))


**4. 装饰器 `@timer`**

装饰器本质是**接收函数 → 返回新函数**的高阶函数,语法糖 `@decorator`。口诀:**装饰器 = 把函数装进壳,`@` 语法糖,`functools.wraps` 留名**。永远写 `@functools.wraps(fn)`,否则被装饰的函数名会变成 `wrapper`,调试/文档全部出错。


In [ ]:
import functools, time

def timer(fn):
    @functools.wraps(fn)        # 保留原函数名/文档
    def wrapper(*args, **kwargs):
        start = time.time()
        result = fn(*args, **kwargs)
        print(f"{fn.__name__} 耗时 {time.time() - start:.4f}s")
        return result
    return wrapper

@timer                          # 等价于 slow = timer(slow)
def slow():
    time.sleep(0.1)

slow()                          # slow 耗时 0.1xxx s
print(slow.__name__)            # slow(被 wraps 保留)


**5. 带参数的装饰器**

需要参数时再加一层:外层接收参数,中层接收函数,内层是真正包装。


In [ ]:
import functools

def log(level="INFO"):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            print(f"[{level}] 调用 {fn.__name__}")
            return fn(*args, **kwargs)
        return wrapper
    return decorator

@log("DEBUG")
def add(a, b):
    return a + b

print(add(1, 2))                # [DEBUG] 调用 add → 3


**今日小结**

import / from / as 三件套;`yield` 生成器惰性产出;`__enter__` / `__exit__` 自定义上下文管理器;`@decorator` + `@functools.wraps` 装饰器。

**练习 1 ⭐⭐**:把 Day08 `Student` 类移到独立模块 `student.py`,主程序 `import` 使用。

**练习 2 ⭐⭐⭐**:生成器实现斐波那契前 n 项。

**练习 3 ⭐⭐⭐⭐**:写一个带重试次数的 @retry(n) 装饰器,失败重试 n 次。

> 🔴 **易错点**:装饰器漏 `@functools.wraps` 原函数名丢失;生成器遍历两次第二轮空;`__exit__` 签名忘加三个 `exc_` 参数报 `TypeError`。
